# Étape 3.6-3.7 — Valider le GPU contre le séquentiel et mesurer le speedup (issue #11)

**Prêt à exécuter : Exécution → Tout exécuter.** Runtime : **GPU T4**.

1. **Validation** : 10 runs du DE séquentiel (`sde`, Lounis) et 10 runs du DE GPU v1 sur Sphere et Rastrigin, D = 10, N = 50. Test de Mann-Whitney : si p > 0,05, on ne peut pas distinguer les deux distributions d'erreurs.
2. **Speedup** : temps séquentiel / temps GPU pour D = 10, 50, 100 avec N = 100, sur les 4 fonctions (5 runs chacun).

Les deux programmes tournent **sur la même machine Colab** (CPU Xeon de Colab, GPU T4), avec la même interface et le même budget.

> Le code séquentiel vient de la branche de Lounis (`lounis/2.7-2.9-csv-convergence`) s'il n'est pas encore dans `main`. Il est seulement compilé ici, jamais ajouté au commit.

In [ ]:
from google.colab import userdata
GH_USER = "amenacer"
REPO    = "DE_CUDA"
BRANCHE = "amina/3.6-3.7-valider-gpu-speedup"
token   = userdata.get("GITHUB_TOKEN")
!rm -rf /content/{REPO}
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO}
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git checkout -q {BRANCHE} && git branch --show-current
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import subprocess, re, statistics as st, os, json
ROOT = "/content/DE_CUDA"
RAPPORT = []                       # tout ce qui est affiche est aussi garde pour le rapport
def note(s=""):
    print(s); RAPPORT.append(str(s))

def run(args, cwd=ROOT, ok_codes=(0,)):
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if p.returncode not in ok_codes:
        print(p.stdout); print(p.stderr)
        raise RuntimeError(f"{args[0]} a renvoye {p.returncode}")
    return p

def compile_cuda(src, out):
    p = run(["nvcc", "-O3", "-arch=sm_75", "-Xcompiler", "-Wall", "-Xcompiler", "-Wextra",
             "-o", out, src], cwd=f"{ROOT}/src/cuda")
    warn = (p.stdout + p.stderr).strip()
    note(f"{src} : " + ("compilation SANS aucun warning" if not warn else "WARNINGS :\n" + warn))

KV = re.compile(r"(\w+)=(\S+)")
def lance(prog, f, D, N, seed, csv=None, extra=()):
    """Lance un run et renvoie le dictionnaire de sa derniere ligne (best_error, time_s, ...)."""
    args = [prog, f, str(D), str(N), str(seed)] + ([csv] if csv else []) + list(extra)
    out = run(args).stdout.strip().splitlines()[-1]
    d = dict(KV.findall(out))
    for k in ("best_error", "time_s"):
        d[k] = float(d[k])
    return d

def mann_whitney(a, b):
    """p-valeur du test de Mann-Whitney bilateral ; 1.0 si les deux series sont identiques et constantes."""
    from scipy.stats import mannwhitneyu
    if len(set(a) | set(b)) == 1:
        return 1.0
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue)

def tableau_md(lignes, entetes):
    s = "| " + " | ".join(entetes) + " |\n|" + "---|" * len(entetes) + "\n"
    for l in lignes:
        s += "| " + " | ".join(str(x) for x in l) + " |\n"
    return s
print("outils prets")

### 1 — Récupérer et compiler les deux programmes

In [ ]:
if not os.path.exists(f"{ROOT}/src/seq/sde.cpp"):
    run(["git", "fetch", "-q", "origin", "lounis/2.7-2.9-csv-convergence"])
    src = run(["git", "show", "FETCH_HEAD:src/seq/sde.cpp"]).stdout
    open(f"{ROOT}/src/seq/sde.cpp", "w").write(src)
    note("sde.cpp recupere depuis la branche lounis/2.7-2.9-csv-convergence (non commite)")
run(["g++", "-std=c++14", "-O2", "-o", "sde", "sde.cpp"], cwd=f"{ROOT}/src/seq")
compile_cuda("de_cuda_v1.cu", "de_cuda_v1")
SDE, V1 = f"{ROOT}/src/seq/sde", f"{ROOT}/src/cuda/de_cuda_v1"
note("CPU : " + run(["bash", "-c", "lscpu | grep 'Model name' | sed 's/Model name: *//'"]).stdout.strip())

### 2 — Validation : 10 runs CPU contre 10 runs GPU, test de Mann-Whitney

In [ ]:
CSV_VAL = f"{ROOT}/results/validation_v1_vs_sde.csv"
os.makedirs(f"{ROOT}/results", exist_ok=True)
if os.path.exists(CSV_VAL): os.remove(CSV_VAL)
lignes = []
for f in ["sphere", "rastrigin"]:
    e = {"sde": [], "cuda_v1": []}
    for seed in range(1, 11):
        e["sde"].append(lance(SDE, f, 10, 50, seed, CSV_VAL)["best_error"])
        e["cuda_v1"].append(lance(V1, f, 10, 50, seed, CSV_VAL)["best_error"])
    p = mann_whitney(e["sde"], e["cuda_v1"])
    for algo in ("sde", "cuda_v1"):
        v = e[algo]
        lignes.append([f, algo, f"{st.mean(v):.3e}", f"{st.pstdev(v):.3e}", f"{st.median(v):.3e}",
                       f"{min(v):.3e}", f"{max(v):.3e}", f"{p:.3f}" if algo == "cuda_v1" else "",
                       ("p > 0,05 : pas de difference" if p > 0.05 else "p <= 0,05 : difference") if algo == "cuda_v1" else ""])
note("Validation D = 10, N = 50, 10 runs (erreur finale f(x) - f(x*))")
note(tableau_md(lignes, ["fonction", "algo", "moyenne", "ecart-type", "mediane", "min", "max", "p (Mann-Whitney)", "conclusion"]))

### 2 bis — Validation plus exigeante (complément)

En D = 10, Sphere et Rastrigin sont résolues exactement (erreur 0) par les deux programmes : le test compare deux séries de zéros et ne peut rien détecter. On ajoute deux cas où l'erreur finale n'est pas nulle : **Rosenbrock D = 10** et **Rastrigin D = 50** (N = 50, 10 runs).

Attention : le séquentiel de Lounis remplace x_i **dès** que l'essai est meilleur (les individus suivants de la même génération voient déjà la nouvelle valeur), alors que le GPU remplace **en fin de génération** (obligatoire en parallèle). Ce sont deux variantes du DE ; une différence de qualité ici peut venir de là, pas d'un bug.

In [ ]:
CSV_VAL2 = f"{ROOT}/results/validation_v1_vs_sde_complement.csv"
if os.path.exists(CSV_VAL2): os.remove(CSV_VAL2)
lignes = []
for f, D in (("rosenbrock", 10), ("rastrigin", 50)):
    e = {"sde": [], "cuda_v1": []}
    for seed in range(1, 11):
        e["sde"].append(lance(SDE, f, D, 50, seed, CSV_VAL2)["best_error"])
        e["cuda_v1"].append(lance(V1, f, D, 50, seed, CSV_VAL2)["best_error"])
    p = mann_whitney(e["sde"], e["cuda_v1"])
    lignes.append([f, D, f"{st.mean(e['sde']):.3e} ± {st.pstdev(e['sde']):.2e}", f"{st.median(e['sde']):.3e}",
                   f"{st.mean(e['cuda_v1']):.3e} ± {st.pstdev(e['cuda_v1']):.2e}", f"{st.median(e['cuda_v1']):.3e}",
                   f"{p:.3f}", "pas de difference" if p > 0.05 else "difference significative"])
note("Validation complementaire (N = 50, 10 runs)")
note(tableau_md(lignes, ["fonction", "D", "sde moyenne ± ET", "sde mediane", "GPU v1 moyenne ± ET", "GPU v1 mediane", "p (Mann-Whitney)", "conclusion (5 %)"]))

### 3 — Speedup : D = 10, 50, 100 avec N = 100 (4 fonctions, 5 runs chacun)

In [ ]:
CSV_SP = f"{ROOT}/results/speedup_v1.csv"
if os.path.exists(CSV_SP): os.remove(CSV_SP)
SP = {}
lignes = []
for f in ["sphere", "rosenbrock", "griewank", "rastrigin"]:
    for D in (10, 50, 100):
        tc = [lance(SDE, f, D, 100, s, CSV_SP)["time_s"] for s in range(1, 6)]
        tg = [lance(V1, f, D, 100, s, CSV_SP)["time_s"] for s in range(1, 6)]
        sp = st.mean(tc) / st.mean(tg)
        SP[(f, D)] = sp
        lignes.append([f, D, f"{st.mean(tc):.4f} ± {st.pstdev(tc):.4f}", f"{st.mean(tg):.4f} ± {st.pstdev(tg):.4f}", f"{sp:.2f}"])
note("Speedup du DE GPU v1 (N = 100, temps moyen sur 5 runs, en secondes)")
note(tableau_md(lignes, ["fonction", "D", "sequentiel (s)", "GPU v1 (s)", "speedup"]))

In [ ]:
import matplotlib.pyplot as plt, numpy as np
os.makedirs(f"{ROOT}/figures", exist_ok=True)
fs = ["sphere", "rosenbrock", "griewank", "rastrigin"]; Ds = [10, 50, 100]
x = np.arange(len(fs)); w = 0.25
fig, ax = plt.subplots(figsize=(7, 3.6))
for k, D in enumerate(Ds):
    ax.bar(x + (k - 1) * w, [SP[(f, D)] for f in fs], w, label=f"D = {D}")
ax.axhline(1, color="gray", lw=0.8, ls="--")
ax.set_xticks(x, fs); ax.set_ylabel("speedup (seq / GPU v1)"); ax.set_title("DE GPU v1, N = 100")
ax.legend(frameon=False); fig.tight_layout()
fig.savefig(f"{ROOT}/figures/speedup_v1.png", dpi=200); plt.show()

### 4 — Sauvegarder et pousser

In [ ]:
os.makedirs(f"{ROOT}/results", exist_ok=True)
with open(f"{ROOT}/results/etape3_validation_speedup.md", "w") as fp:
    fp.write("# Étape 3.6-3.7 — validation du GPU contre le séquentiel et speedup (issue #11)\n\n")
    fp.write("GPU : " + run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).stdout.strip() + "\n\n")
    fp.write("```\n" + "\n".join(RAPPORT) + "\n```\n")
%cd /content/DE_CUDA
!git add results/validation_v1_vs_sde.csv results/validation_v1_vs_sde_complement.csv results/speedup_v1.csv figures/speedup_v1.png results/etape3_validation_speedup.md
!git status --short
!git commit -q -m "3.6-3.7 : validation GPU v1 contre le sequentiel et speedup (refs #11)" && git push -q origin {BRANCHE} && echo "=> pousse sur {BRANCHE}"